# Diagnóstico: Confusiones, Aumentaciones y Modelos Mejorados

**Estructura:**
1. Setup y carga de datos
2. ¿Qué glosas se confunden más? (análisis de errores)
3. ¿Las aumentaciones tienen sentido? (sanity check visual)
4. Modelos mejorados: kNN con descriptor de trayectoria + PointNet con atención temporal
5. Comparación final

## 1. Setup

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 10})

import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

from dataset import split_dataset, collate_fn, TMAX, INPUT_DIM, N_LANDMARKS
from augmentations import LandmarkAugmenter, AimCLRViewGenerator

PARQUET_PATH = "../corpus_LSM_esp/lsm_dataset.parquet"
SEED         = 42
print("Setup OK")

Setup OK


In [2]:
train_ds, val_ds, test_ds = split_dataset(
    PARQUET_PATH, train_ratio=0.70, val_ratio=0.15,
    seed=SEED, score_thresh=0.3,
)
NUM_CLASSES = train_ds.num_classes
IDX2GLOSA   = train_ds.idx2glosa
GLOSA2IDX   = train_ds.glosa2idx

def ds_to_numpy(ds):
    """LSMDataset → X (N, TMAX, 133, 2), y (N,)"""
    X, y = [], []
    for item in ds:
        T   = item["T"]
        kpt = item["keypoints"][:T].numpy().reshape(T, N_LANDMARKS, 2)
        if T < TMAX:
            pad = np.zeros((TMAX - T, N_LANDMARKS, 2), dtype=np.float32)
            kpt = np.concatenate([kpt, pad], axis=0)
        X.append(kpt)
        y.append(item["label"].item())
    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)

print("Convirtiendo a numpy...")
X_train, y_train = ds_to_numpy(train_ds)
X_test,  y_test  = ds_to_numpy(test_ds)
print(f"X_train {X_train.shape} | X_test {X_test.shape}")

[split_dataset] Cargando ../corpus_LSM_esp/lsm_dataset.parquet …
[split_dataset] 2447 videos | 249 glosas únicas
[split_dataset] train=1718 | val=402 | test=327
Convirtiendo a numpy...
X_train (1718, 200, 133, 2) | X_test (327, 200, 133, 2)


---
## 2. ¿Qué glosas se confunden más?

Primero entrenamos el PointNet base para tener predicciones sobre el test set.

In [3]:
from pointcloud_models import PointNetTrainer, DistanceModel

# Entrenar kNN y PointNet base (los mismos que ya tienes)
knn = DistanceModel(k=5)
knn.fit(X_train, y_train)
knn_preds  = knn.predict(X_test)
knn_probas = knn.predict_proba(X_test)

pn = PointNetTrainer(num_classes=NUM_CLASSES, epochs=50, patience=10)
pn.fit(X_train, y_train)
pn_preds  = pn.predict(X_test)
pn_probas = pn.predict_proba(X_test)

print(f"kNN   Top-1: {accuracy_score(y_test, knn_preds):.4f}")
print(f"PointNet Top-1: {accuracy_score(y_test, pn_preds):.4f}")

[DistanceModel] Extrayendo descriptores de 1718 muestras...
[DistanceModel] Descriptor dim: 63
[DistanceModel] Entrenado. k=5, clases=249
[PointNetTrainer] Preparando datos (1718 muestras)...
[PointNetTrainer] Entrenando en cuda...
 Epoch |     Loss |    Acc
------------------------------
     1 |   5.5249 |  0.005
     5 |   4.7708 |  0.042
    10 |   3.9088 |  0.166
    15 |   3.3361 |  0.306
    20 |   3.0191 |  0.381
    25 |   2.8000 |  0.421
    30 |   2.6034 |  0.508
    35 |   2.5193 |  0.520
    40 |   2.4462 |  0.561
    45 |   2.3798 |  0.590
    50 |   2.3852 |  0.576
[PointNetTrainer] Entrenamiento completo. Mejor loss: 2.3798
kNN   Top-1: 0.0367
PointNet Top-1: 0.2722


In [ ]:
def error_analysis(y_true, y_pred, idx2glosa, model_name, top_n=20):
    """
    Muestra:
    - Glosas con peor accuracy individual
    - Pares de confusión más frecuentes
    - Distribución de confianza en errores vs aciertos
    """
    print(f"\n{'='*55}")
    print(f"  Análisis de errores: {model_name}")
    print(f"{'='*55}")

    # Accuracy por clase
    per_class = {}
    for cls in np.unique(y_true):
        mask = y_true == cls
        acc  = (y_pred[mask] == cls).mean()
        per_class[idx2glosa[cls]] = (acc, mask.sum())

    # Glosas con peor accuracy (mínimo 2 muestras en test)
    worst = sorted(
        [(g, a, n) for g, (a, n) in per_class.items() if n >= 2],
        key=lambda x: x[1]
    )[:top_n]

    print(f"\nPeores {top_n} glosas por accuracy:")
    df_worst = pd.DataFrame(worst, columns=["Glosa", "Accuracy", "N_test"])
    df_worst["Accuracy"] = df_worst["Accuracy"].map("{:.3f}".format)
    display(df_worst)

    # Pares de confusión más frecuentes
    errors = [
        (idx2glosa[t], idx2glosa[p])
        for t, p in zip(y_true, y_pred) if t != p
    ]
    print(f"\nTop 15 pares de confusión (real → predicho):")
    df_conf = pd.DataFrame(
        Counter(errors).most_common(15),
        columns=["(Real → Predicho)", "Frecuencia"]
    )
    display(df_conf)

    return per_class


pc_knn = error_analysis(y_test, knn_preds,  IDX2GLOSA, "DistanceModel (kNN)")
pc_pn  = error_analysis(y_test, pn_preds,   IDX2GLOSA, "PointNetModel")

In [ ]:
# Gráfica: accuracy por glosa (distribución)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (name, pc) in zip(axes, [("kNN", pc_knn), ("PointNet", pc_pn)]):
    accs = [a for g, (a, n) in pc.items() if n >= 2]
    ax.hist(accs, bins=20, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(accs), color="red", linestyle="--",
               label=f"Media={np.mean(accs):.3f}")
    ax.set_title(f"{name} — Distribución accuracy por glosa")
    ax.set_xlabel("Accuracy")
    ax.set_ylabel("Nº glosas")
    ax.legend()

plt.tight_layout()
plt.savefig("accuracy_por_glosa.png", bbox_inches="tight")
plt.show()

In [ ]:
# Matriz de confusión — top 25 glosas más frecuentes en test
from sklearn.metrics import confusion_matrix

def plot_cm(y_true, y_pred, idx2glosa, title, top_n=25):
    unique, counts = np.unique(y_true, return_counts=True)
    top_cls = unique[np.argsort(counts)[::-1][:top_n]]
    mask    = np.isin(y_true, top_cls)
    cm      = confusion_matrix(y_true[mask], y_pred[mask], labels=top_cls)
    # Normalizar por fila para ver tasa de error
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    labels  = [idx2glosa[c] for c in top_cls]

    fig, ax = plt.subplots(figsize=(13, 11))
    sns.heatmap(cm_norm, annot=False, cmap="Blues", vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, ax=ax, linewidths=0.3)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_title(f"{title} — Matriz de confusión normalizada (top {top_n})")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    plt.savefig(f"cm_{title.replace(' ','_').lower()}.png", bbox_inches="tight")
    plt.show()

plot_cm(y_test, knn_preds, IDX2GLOSA, "kNN")
plot_cm(y_test, pn_preds,  IDX2GLOSA, "PointNet")

---
## 3. Sanity check de aumentaciones

Para cada aumentación verificamos:
- ¿Cambia la señal lo suficiente para ser útil?
- ¿La cambia demasiado y rompe la semántica?
- ¿Las coordenadas siguen siendo anatómicamente coherentes?

Usamos una muestra real del dataset.

In [ ]:
import io
import pandas as _pd

# Cargar una muestra cruda (con score) directamente del parquet
df_raw = _pd.read_parquet(PARQUET_PATH)
sample_row = df_raw.iloc[0]
kpts_raw   = np.load(io.BytesIO(sample_row["keypoints"]))  # (T, 133, 3)
T_orig     = kpts_raw.shape[0]

print(f"Glosa: {sample_row['glosa']}")
print(f"Frames: {T_orig}")
print(f"Score promedio: {kpts_raw[:,:,2].mean():.3f}")
print(f"X range: [{kpts_raw[:,:,0].min():.2f}, {kpts_raw[:,:,0].max():.2f}]")
print(f"Y range: [{kpts_raw[:,:,1].min():.2f}, {kpts_raw[:,:,1].max():.2f}]")

In [ ]:
from augmentations import (
    _vary_build, _vary_hand_size, _add_gaussian_noise,
    _temporal_jitter, _speed_perturbation,
    _temporal_flip, _axis_mask, _temporal_blur,
    LandmarkAugmenter, AimCLRViewGenerator,
)
from dataset import preprocess

# Preprocesar muestra base
kpts_base = preprocess(kpts_raw)   # (T, 133, 2) normalizado

def trayectoria_muneca(kpts_xy, wrist_idx=91):
    """Extrae trayectoria x,y de la muñeca izquierda: (T, 2)"""
    return kpts_xy[:, wrist_idx, :]

def plot_aug_comparison(original, augmented, aug_name, wrist=91):
    """Compara la trayectoria de la muñeca entre original y aumentado."""
    orig_traj = original[:, wrist, :]    # (T, 2)
    aug_traj  = augmented[:, wrist, :]   # (T, 2)

    diff = np.abs(aug_traj - orig_traj).mean()
    rel  = diff / (np.abs(orig_traj).mean() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))

    # Trayectoria espacial
    axes[0].plot(orig_traj[:, 0], orig_traj[:, 1], "b-o", ms=2, label="Original", alpha=0.7)
    axes[0].plot(aug_traj[:, 0],  aug_traj[:, 1],  "r-o", ms=2, label=aug_name,   alpha=0.7)
    axes[0].set_title("Trayectoria muñeca izq (espacio)")
    axes[0].legend(fontsize=7)
    axes[0].set_aspect("equal")

    # X en tiempo
    axes[1].plot(orig_traj[:, 0], "b", label="Original", alpha=0.8)
    axes[1].plot(aug_traj[:, 0],  "r", label=aug_name,   alpha=0.8)
    axes[1].set_title("X(t) muñeca")
    axes[1].legend(fontsize=7)

    # Y en tiempo
    axes[2].plot(orig_traj[:, 1], "b", label="Original", alpha=0.8)
    axes[2].plot(aug_traj[:, 1],  "r", label=aug_name,   alpha=0.8)
    axes[2].set_title("Y(t) muñeca")
    axes[2].legend(fontsize=7)

    fig.suptitle(
        f"{aug_name}  |  Δ_abs={diff:.4f}  Δ_rel={rel:.2%}  "
        f"{'✓ razonable' if 0.01 < rel < 0.25 else '⚠ revisar'}",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(f"aug_{aug_name.replace(' ','_')}.png", bbox_inches="tight")
    plt.show()
    print(f"  Δ_abs={diff:.4f}  Δ_rel={rel:.2%}")

print("Funciones de diagnóstico cargadas.")

In [ ]:
# ── A. vary_build ─────────────────────────────────────────────────────────
# Esperado: desplazamiento lateral pequeño de los brazos (~5-10%)
# Problema si: mueve los dedos/cara
aug_build = preprocess(_vary_build(kpts_raw.copy(), s=1.10))
plot_aug_comparison(kpts_base, aug_build, "vary_build s=1.10")

In [ ]:
# ── B. vary_hand_size ──────────────────────────────────────────────────────
# Esperado: escala dedos alrededor de la muñeca (~2-4%)
# Problema si: mueve la muñeca misma
aug_hand = preprocess(_vary_hand_size(kpts_raw.copy(), s=1.05))
plot_aug_comparison(kpts_base, aug_hand, "vary_hand_size s=1.05", wrist=91)

# Verificar que la muñeca NO se movió
delta_wrist = np.abs(aug_hand[:, 91, :] - kpts_base[:, 91, :]).mean()
print(f"  Delta muñeca (debe ser ~0): {delta_wrist:.6f}")

In [ ]:
# ── C. gaussian_noise ─────────────────────────────────────────────────────
# Esperado: perturbación pequeña uniforme (~1-2%)
# Problema si: sigma demasiado grande distorsiona la forma
for sigma in [0.005, 0.015, 0.03]:
    aug_noise = preprocess(_add_gaussian_noise(kpts_raw.copy(), sigma=sigma))
    plot_aug_comparison(kpts_base, aug_noise, f"noise σ={sigma}")

In [ ]:
# ── D. speed_perturbation ─────────────────────────────────────────────────
# Esperado: misma forma espacial, diferente velocidad
# Crítico: la trayectoria espacial debe ser similar, solo comprimida/expandida en t
for rate in [0.8, 1.0, 1.2]:
    from augmentations import _speed_perturbation
    aug_speed = preprocess(_speed_perturbation(kpts_raw.copy(), rate=rate))
    plot_aug_comparison(kpts_base, aug_speed, f"speed rate={rate}")

In [ ]:
# ── E. temporal_flip (AimCLR) ────────────────────────────────────────────
# Esperado: la seña al revés — SÍ rompe semántica, pero es una vista extrema
# Es válido para AimCLR (fuerza invarianza), NO para aumentación estándar
aug_flip = preprocess(_temporal_flip(kpts_raw.copy()))
plot_aug_comparison(kpts_base, aug_flip, "temporal_flip")
print("⚠ temporal_flip SÍ rompe la semántica temporal.")
print("  Es válido SOLO para AimCLR (vista contrastiva), no para entrenamiento estándar.")

In [ ]:
# ── F. axis_mask (AimCLR) ────────────────────────────────────────────────
# Esperado: elimina toda la información de un eje — muy agresivo
# Válido para AimCLR, problemático como augmentación estándar
aug_axis = preprocess(_axis_mask(kpts_raw.copy(), axis=0))  # elimina X
plot_aug_comparison(kpts_base, aug_axis, "axis_mask X=0")
print("⚠ axis_mask elimina TODA la información de un eje.")
print("  Con p=0.3 en AimCLR puede ser demasiado agresivo — revisar si colapsa la pérdida D3M.")

In [ ]:
# ── G. temporal_blur (AimCLR) ────────────────────────────────────────────
# Esperado: suaviza movimientos rápidos, mantiene la forma general
for sigma in [1.0, 1.5, 3.0]:
    aug_blur = preprocess(_temporal_blur(kpts_raw.copy(), sigma=sigma))
    plot_aug_comparison(kpts_base, aug_blur, f"temporal_blur σ={sigma}")

In [ ]:
# ── H. Pipeline completo LandmarkAugmenter ────────────────────────────────
# Verificar que el pipeline combinado no distorsiona demasiado
augmenter = LandmarkAugmenter()
diffs = []
for _ in range(50):
    aug = preprocess(augmenter(kpts_raw.copy()))
    diff = np.abs(aug[:, 91, :] - kpts_base[:, 91, :]).mean()  # muñeca
    diffs.append(diff)

print(f"LandmarkAugmenter — Δ_abs muñeca izq (50 muestras):")
print(f"  Media: {np.mean(diffs):.4f}")
print(f"  P95  : {np.percentile(diffs, 95):.4f}")
print(f"  Max  : {np.max(diffs):.4f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(diffs, bins=15, color="#55A868", edgecolor="white")
ax.set_title("Distribución de perturbación (LandmarkAugmenter, 50 runs)")
ax.set_xlabel("Δ_abs muñeca izq")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

# Veredicto automático
if np.mean(diffs) < 0.02:
    print("✓ Perturbación muy conservadora — puede no ayudar con el overfitting")
elif np.mean(diffs) < 0.08:
    print("✓ Perturbación razonable")
else:
    print("⚠ Perturbación alta — puede corromper glosas dinámicas")

---
## 4. Modelos mejorados

Basado en el análisis:
- **kNN mejorado**: descriptor con énfasis en trayectoria temporal (DTW-like features)
- **PointNet mejorado**: GRU encima del max-pooling para capturar orden temporal

In [ ]:
# ══════════════════════════════════════════════════════════════
# kNN MEJORADO — descriptor centrado en trayectoria temporal
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

_WRIST_L_I = 91
_WRIST_R_I = 112
_TIPS_L    = [96, 100, 104, 108, 112]   # puntas dedos mano izq (aprox)
_TIPS_R    = [117, 121, 125, 129, 133-1] # puntas dedos mano der (aprox)

def trajectory_descriptor(seq: np.ndarray) -> np.ndarray:
    """
    Descriptor centrado en la trayectoria temporal.
    seq: (T, 133, 2)  — ya normalizado, sin padding (o con él, se ignoran ceros)

    Extrae para cada punto de interés (muñecas, puntas de dedos):
      - Trayectoria cuantizada en K segmentos (captura el "dibujo" del movimiento)
      - Velocidad frame a frame: media, std, max, percentiles
      - Aceleración: media, std
      - Longitud total de trayectoria (arclength)
      - Bounding box de la trayectoria (x_range, y_range)
      - Posición relativa inicio→fin (desplazamiento neto)
    """
    T = seq.shape[0]
    feats = []

    # Keypoints de interés: muñecas + puntas de dedos
    key_pts = [_WRIST_L_I, _WRIST_R_I] + _TIPS_L[:3] + _TIPS_R[:3]

    for kp in key_pts:
        traj = seq[:, kp, :]                         # (T, 2)

        # Trayectoria cuantizada: 10 puntos equiespaciados
        idx  = np.linspace(0, T-1, 10).astype(int)
        feats.append(traj[idx].ravel())               # (20,)

        # Velocidad
        vel   = np.diff(traj, axis=0)                # (T-1, 2)
        speed = np.linalg.norm(vel, axis=-1)          # (T-1,)
        if len(speed) > 0:
            feats.append(np.array([
                speed.mean(), speed.std(),
                speed.max(),
                np.percentile(speed, 25),
                np.percentile(speed, 75),
            ]))
            # Aceleración
            acc = np.diff(speed)
            feats.append(np.array([
                acc.mean() if len(acc) > 0 else 0.0,
                acc.std()  if len(acc) > 0 else 0.0,
            ]))
        else:
            feats.append(np.zeros(5))
            feats.append(np.zeros(2))

        # Arclength (longitud total del camino)
        arclength = np.sum(np.linalg.norm(vel, axis=-1)) if T > 1 else 0.0
        feats.append(np.array([arclength]))

        # Bounding box
        feats.append(np.array([
            traj[:, 0].max() - traj[:, 0].min(),
            traj[:, 1].max() - traj[:, 1].min(),
        ]))

        # Desplazamiento neto inicio→fin
        feats.append(traj[-1] - traj[0])             # (2,)

    # Relación espacial entre ambas muñecas a lo largo del tiempo
    rel_wrists = seq[:, _WRIST_R_I, :] - seq[:, _WRIST_L_I, :]  # (T, 2)
    idx = np.linspace(0, T-1, 8).astype(int)
    feats.append(rel_wrists[idx].ravel())            # (16,)
    feats.append(np.array([
        np.linalg.norm(rel_wrists, axis=-1).mean(),
        np.linalg.norm(rel_wrists, axis=-1).std(),
    ]))

    return np.concatenate([f.ravel() for f in feats]).astype(np.float32)


def build_trajectory_descriptors(X: np.ndarray) -> np.ndarray:
    """X: (N, T, 133, 2) → (N, D)"""
    return np.stack([trajectory_descriptor(x) for x in X])


print("Descriptor de trayectoria definido.")
# Verificar dimensión
d_test = trajectory_descriptor(X_train[0])
print(f"Dimensión del descriptor: {d_test.shape[0]}")

In [ ]:
# Entrenar kNN mejorado
print("Extrayendo descriptores de trayectoria...")
D_train = build_trajectory_descriptors(X_train)
D_test  = build_trajectory_descriptors(X_test)
print(f"Descriptores: train {D_train.shape} | test {D_test.shape}")

knn_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=7,
        metric="euclidean",
        weights="distance",
        algorithm="ball_tree",
        n_jobs=-1,
    )),
])
knn_v2.fit(D_train, y_train)

knn_v2_preds  = knn_v2.predict(D_test)
knn_v2_probas = knn_v2.predict_proba(D_test)

knn_v2_top1 = accuracy_score(y_test, knn_v2_preds)
knn_v2_top3 = top_k_accuracy_score(y_test, knn_v2_probas, k=3,
                                    labels=list(range(NUM_CLASSES)))
knn_v2_f1   = f1_score(y_test, knn_v2_preds, average="macro",
                        labels=list(range(NUM_CLASSES)), zero_division=0)

print(f"\nkNN v2 (trayectoria)")
print(f"  Top-1: {knn_v2_top1:.4f}  (antes: {accuracy_score(y_test, knn_preds):.4f})")
print(f"  Top-3: {knn_v2_top3:.4f}")
print(f"  F1   : {knn_v2_f1:.4f}")

In [ ]:
# Fix MIOpen (ROCm/AMD): necesita directorio temporal para compilar kernels RNN
# Si usas NVIDIA este bloque no tiene efecto.
import os, tempfile

miopen_cache = os.path.expanduser('~/.config/miopen')
tmp_dir      = '/tmp/miopen_tmp'
os.makedirs(miopen_cache, exist_ok=True)
os.makedirs(tmp_dir,      exist_ok=True)

os.environ['MIOPEN_USER_DB_PATH']   = miopen_cache
os.environ['TMPDIR']                = tmp_dir
os.environ['MIOPEN_DISABLE_CACHE']  = '0'
# Deshabilitar fusión de kernels que falla sin /tmp
os.environ['MIOPEN_DISABLE_FUSIONS'] = '1'

print(f'MIOpen cache : {miopen_cache}')
print(f'TMPDIR       : {tmp_dir}')
print('Fix aplicado.')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class GRUCell(nn.Module):
    """
    GRU implementado con operaciones lineales puras (sin _VF.gru).
    Evita el kernel de MIOpen que falla en algunas configs ROCm.
    Funcionalmente idéntico al nn.GRU estándar.
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        # Puerta de reset, update y nueva activación — todo en una matriz
        self.linear_z = nn.Linear(input_size + hidden_size, hidden_size)
        self.linear_r = nn.Linear(input_size + hidden_size, hidden_size)
        self.linear_n = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """x: (B, input_size), h: (B, hidden_size) → h_new: (B, hidden_size)"""
        xh = torch.cat([x, h], dim=-1)
        z  = torch.sigmoid(self.linear_z(xh))
        r  = torch.sigmoid(self.linear_r(xh))
        xrh = torch.cat([x, r * h], dim=-1)
        n  = torch.tanh(self.linear_n(xrh))
        return (1 - z) * h + z * n


class ManualBiGRU(nn.Module):
    """
    GRU bidireccional implementado manualmente frame a frame.
    Más lento que nn.GRU pero no depende de MIOpen/cuDNN para compilar kernels.
    """
    def __init__(self, input_size: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Capas forward y backward para cada layer
        self.fwd_cells = nn.ModuleList()
        self.bwd_cells = nn.ModuleList()
        for l in range(num_layers):
            in_size = input_size if l == 0 else hidden_size * 2
            self.fwd_cells.append(GRUCell(in_size, hidden_size))
            self.bwd_cells.append(GRUCell(in_size, hidden_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x   : (B, T, input_size)
        out : (B, T, hidden_size * 2)
        """
        B, T, _ = x.shape
        h = x

        for l in range(self.num_layers):
            h_fwd = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
            h_bwd = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
            fwd_out = []
            bwd_out = []

            for t in range(T):
                h_fwd = self.fwd_cells[l](h[:, t, :], h_fwd)
                fwd_out.append(h_fwd)

            for t in reversed(range(T)):
                h_bwd = self.bwd_cells[l](h[:, t, :], h_bwd)
                bwd_out.insert(0, h_bwd)

            fwd_tensor = torch.stack(fwd_out, dim=1)   # (B, T, H)
            bwd_tensor = torch.stack(bwd_out, dim=1)   # (B, T, H)
            h = torch.cat([fwd_tensor, bwd_tensor], dim=-1)  # (B, T, 2H)

        return h


class PointNetGRU(nn.Module):
    """
    PointNet + BiGRU manual (sin dependencia de MIOpen kernel compilation).
    Arquitectura idéntica a la original pero usa ManualBiGRU en lugar de nn.GRU.
    """
    def __init__(
        self,
        num_classes: int   = 249,
        input_dim:   int   = 266,
        latent_dim:  int   = 128,
        gru_hidden:  int   = 256,
        gru_layers:  int   = 2,
        dropout:     float = 0.3,
    ):
        super().__init__()
        self.frame_mlp = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
        )
        self.gru = ManualBiGRU(latent_dim, gru_hidden, num_layers=gru_layers)
        gru_out_dim = gru_hidden * 2

        self.attn = nn.Sequential(
            nn.Linear(gru_out_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(gru_out_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor, valid_mask: torch.Tensor = None) -> torch.Tensor:
        B, T, _ = x.shape
        h = self.frame_mlp(x.reshape(B * T, -1)).reshape(B, T, -1)
        h = self.gru(h)
        attn_scores = self.attn(h).squeeze(-1)
        if valid_mask is not None:
            attn_scores = attn_scores.masked_fill(~valid_mask, float("-inf"))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = (h * attn_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(context)


print("PointNetGRU (ManualBiGRU) definido.")
_m = PointNetGRU(num_classes=NUM_CLASSES).to(DEVICE)
_x = torch.randn(4, TMAX, 266).to(DEVICE)
_o = _m(_x)
print(f"Output shape: {_o.shape}  ✓")
del _m, _x, _o


In [ ]:
# Preparar tensores
def numpy_to_tensors(X, y, tmax=TMAX):
    """X: (N, T, 133, 2) → frames (N, T, 266), valid_mask (N, T), labels (N,)"""
    N = len(X)
    frames = torch.from_numpy(X.reshape(N, tmax, -1))   # (N, T, 266)
    valid  = (frames.abs().sum(dim=-1) > 0)              # True donde no es padding
    labels = torch.from_numpy(y)
    return frames, valid, labels

f_train, v_train, l_train = numpy_to_tensors(X_train, y_train)
f_test,  v_test,  l_test  = numpy_to_tensors(X_test,  y_test)

train_loader = DataLoader(
    TensorDataset(f_train, v_train, l_train),
    batch_size=32, shuffle=True, drop_last=False,
)
print(f"Train loader: {len(train_loader)} batches")

In [ ]:
# Entrenamiento PointNetGRU
model_v2 = PointNetGRU(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model_v2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_acc   = 0.0
best_state = None
patience_counter = 0
PATIENCE = 15

print(f"{'Ep':>4} | {'Loss':>7} | {'Train Acc':>9} | {'Note'}")
print("-" * 40)

for epoch in range(1, 81):
    model_v2.train()
    total_loss, correct, total = 0.0, 0, 0

    for frames_b, valid_b, labels_b in train_loader:
        frames_b = frames_b.to(DEVICE)
        valid_b  = valid_b.to(DEVICE)
        labels_b = labels_b.to(DEVICE)

        optimizer.zero_grad()
        logits = model_v2(frames_b, valid_b)
        loss   = criterion(logits, labels_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(-1) == labels_b).sum().item()
        total      += len(labels_b)

    scheduler.step()
    acc = correct / total
    avg_loss = total_loss / len(train_loader)

    note = ""
    if acc > best_acc:
        best_acc   = acc
        best_state = {k: v.clone() for k, v in model_v2.state_dict().items()}
        patience_counter = 0
        note = "✓ best"
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping en época {epoch}.")
            break

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>4d} | {avg_loss:>7.4f} | {acc:>9.4f} | {note}")

model_v2.load_state_dict(best_state)
print(f"\nMejor train acc: {best_acc:.4f}")

In [ ]:
# Evaluar PointNetGRU en test
model_v2.eval()
with torch.no_grad():
    logits_test = model_v2(
        f_test.to(DEVICE),
        v_test.to(DEVICE),
    ).cpu()

pn_v2_probas = F.softmax(logits_test, dim=-1).numpy()
pn_v2_preds  = pn_v2_probas.argmax(axis=-1)

pn_v2_top1 = accuracy_score(y_test, pn_v2_preds)
pn_v2_top3 = top_k_accuracy_score(y_test, pn_v2_probas, k=3,
                                   labels=list(range(NUM_CLASSES)))
pn_v2_f1   = f1_score(y_test, pn_v2_preds, average="macro",
                       labels=list(range(NUM_CLASSES)), zero_division=0)

print(f"PointNetGRU v2")
print(f"  Top-1: {pn_v2_top1:.4f}  (antes: {accuracy_score(y_test, pn_preds):.4f})")
print(f"  Top-3: {pn_v2_top3:.4f}")
print(f"  F1   : {pn_v2_f1:.4f}")

---
## 5. Comparación final

In [ ]:
rows = [
    {"Modelo": "kNN v1 (descriptor estadístico)",
     "Top-1": accuracy_score(y_test, knn_preds),
     "Top-3": top_k_accuracy_score(y_test, knn_probas, k=3, labels=list(range(NUM_CLASSES))),
     "F1":    f1_score(y_test, knn_preds, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)},
    {"Modelo": "kNN v2 (descriptor trayectoria)",
     "Top-1": knn_v2_top1, "Top-3": knn_v2_top3, "F1": knn_v2_f1},
    {"Modelo": "PointNet v1 (max-pooling)",
     "Top-1": accuracy_score(y_test, pn_preds),
     "Top-3": top_k_accuracy_score(y_test, pn_probas, k=3, labels=list(range(NUM_CLASSES))),
     "F1":    f1_score(y_test, pn_preds, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)},
    {"Modelo": "PointNetGRU v2 (GRU + atención)",
     "Top-1": pn_v2_top1, "Top-3": pn_v2_top3, "F1": pn_v2_f1},
]

df_cmp = pd.DataFrame(rows).set_index("Modelo")
display(df_cmp.style.format("{:.4f}").highlight_max(axis=0, color="#c6efce"))

# Gráfica
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_cmp))
w = 0.25
ax.bar(x - w, df_cmp["Top-1"], w, label="Top-1",   color="#4C72B0")
ax.bar(x,     df_cmp["Top-3"], w, label="Top-3",   color="#55A868")
ax.bar(x + w, df_cmp["F1"],    w, label="Macro F1", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels(df_cmp.index, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 1.0)
ax.legend()
ax.yaxis.grid(True, alpha=0.3)
ax.set_title("Comparación de modelos v1 vs v2")
plt.tight_layout()
plt.savefig("comparacion_v1_v2.png", bbox_inches="tight")
plt.show()

In [ ]:
# Análisis de errores del mejor modelo
best_preds = pn_v2_preds if pn_v2_top1 >= knn_v2_top1 else knn_v2_preds
best_name  = "PointNetGRU v2" if pn_v2_top1 >= knn_v2_top1 else "kNN v2"

print(f"Mejor modelo simple: {best_name}")
error_analysis(y_test, best_preds, IDX2GLOSA, best_name)
plot_cm(y_test, best_preds, IDX2GLOSA, best_name)